# ⚙️ Obesity Prediction Model - Feature Engineering

## 📋 Notebook Overview
This notebook handles advanced feature engineering, scaling, and selection for the obesity prediction model.

### 🎯 Objectives:
- Load preprocessed data from previous stage
- Create engineered features (BMI, etc.)
- Apply feature scaling and normalization
- Perform feature selection using multiple techniques
- Prepare final dataset for model training

### 📁 Input/Output:
- **Input**: `../data/preprocessed_data.csv`
- **Output**: `../data/engineered_features.csv`, feature artifacts

## 📚 Import Required Libraries

In [1]:
# Import libraries for feature engineering
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    mutual_info_classif, 
    f_classif, 
    SelectKBest, 
    RFE
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from pathlib import Path
import joblib
import json
import warnings

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 📂 Load Preprocessed Data

In [2]:
# Load the preprocessed data
data_path = Path("../data/preprocessed_data.csv")
target_info_path = Path("../data/target_classes.json")

if data_path.exists() and target_info_path.exists():
    df = pd.read_csv(data_path)
    
    with open(target_info_path, 'r') as f:
        target_info = json.load(f)
    
    print(f"✅ Data loaded successfully!")
    print(f"📊 Dataset shape: {df.shape}")
    print(f"🎯 Target classes: {len(target_info['target_classes'])}")
    print(f"📋 Target classes: {target_info['target_classes']}")
else:
    print(f"❌ Error: Required files not found")
    print(f"   Data file: {data_path.exists()}")
    print(f"   Target info: {target_info_path.exists()}")
    print("Please run previous notebooks first.")

✅ Data loaded successfully!
📊 Dataset shape: (20758, 48)
🎯 Target classes: 7
📋 Target classes: ['Overweight_Level_II', 'Normal_Weight', 'Insufficient_Weight', 'Obesity_Type_III', 'Obesity_Type_II', 'Overweight_Level_I', 'Obesity_Type_I']


## 🔬 Create Engineered Features

Let's create meaningful derived features that can improve model performance.

### 📏 BMI Feature Creation

In [3]:
# Create a copy for feature engineering
df_engineered = df.copy()

# Keep track of new engineered features
new_features = []

# Calculate BMI (Body Mass Index)
df_engineered['BMI'] = (df_engineered['Weight'] / (df_engineered['Height'] ** 2)).round(2)
new_features.append('BMI')

print("✅ BMI feature created successfully!")
print(f"📊 BMI statistics:")
print(df_engineered['BMI'].describe())

# Visualize BMI distribution
fig_bmi = px.histogram(
    df_engineered, 
    x='BMI', 
    title='📏 BMI Distribution',
    nbins=30,
    color_discrete_sequence=['lightblue']
)
fig_bmi.update_layout(height=400, title_x=0.5)
fig_bmi.show()

print(f"\n🔧 New features created: {new_features}")

✅ BMI feature created successfully!
📊 BMI statistics:
count    20758.000000
mean        30.241760
std          8.333982
min         12.870000
25%         24.090000
50%         29.380000
75%         37.010000
max         55.000000
Name: BMI, dtype: float64



🔧 New features created: ['BMI']


In [4]:
# BMI vs Obesity Type relationship
# Find the correct encoded column name
encoded_columns = [col for col in df_engineered.columns if 'encoded' in col.lower()]
target_col = encoded_columns[0] if encoded_columns else 'NObeyesdad_encoded'
target_classes = target_info['target_classes']

print(f"✅ Using target column: {target_col}")

# Create target labels for visualization
df_viz = df_engineered.copy()
df_viz['Obesity_Type'] = df_viz[target_col].map({i: target_classes[i] for i in range(len(target_classes))})

fig_bmi_obesity = px.box(
    df_viz,
    x='Obesity_Type',
    y='BMI',
    title='📏 BMI Distribution by Obesity Type',
    color='Obesity_Type'
)
fig_bmi_obesity.update_layout(
    height=600,
    title_x=0.5,
    xaxis_tickangle=45
)
fig_bmi_obesity.show()

print(f"\n📊 BMI by Obesity Type (Mean values):")
bmi_by_type = df_viz.groupby('Obesity_Type')['BMI'].mean().sort_values()
for obesity_type, avg_bmi in bmi_by_type.items():
    print(f"   {obesity_type:<25}: {avg_bmi:.2f}")

✅ Using target column: NObeyesdad_encoded



📊 BMI by Obesity Type (Mean values):
   Overweight_Level_II      : 17.58
   Normal_Weight            : 22.00
   Overweight_Level_I       : 26.06
   Obesity_Type_I           : 28.19
   Insufficient_Weight      : 32.15
   Obesity_Type_III         : 36.52
   Obesity_Type_II          : 41.78


## 🎯 Feature Selection Analysis

Let's analyze which features are most important for predicting obesity.

### 📊 Prepare Features and Target

In [5]:
# Separate features and target
# Exclude both original and encoded target columns from features
target_columns_to_exclude = [target_col, 'NObeyesdad']  # Both encoded and original target
X = df_engineered.drop(columns=target_columns_to_exclude, errors='ignore')
y = df_engineered[target_col]

print(f"✅ Features and target separated:")
print(f"   Features shape: {X.shape}")
print(f"   Target shape: {y.shape}")
print(f"   Target classes: {y.nunique()}")
print(f"   Excluded columns: {target_columns_to_exclude}")

# Display feature types
print(f"\n📊 Feature types:")
print(X.dtypes.value_counts())

# Check that target columns are properly excluded
if 'NObeyesdad' in X.columns or target_col in X.columns:
    print("⚠️ Warning: Target columns still present in features!")
else:
    print("✅ Target columns properly excluded from features")

✅ Features and target separated:
   Features shape: (20758, 47)
   Target shape: (20758,)
   Target classes: 7
   Excluded columns: ['NObeyesdad_encoded', 'NObeyesdad']

📊 Feature types:
bool       22
float64    17
object      8
Name: count, dtype: int64
✅ Target columns properly excluded from features


### 🧠 Mutual Information Analysis

In [6]:
# Check for object/string columns that need encoding
print("🔍 Checking data types in X:")
print(X.dtypes.value_counts())

print("\n📋 Object columns found:")
object_cols = X.select_dtypes(include=['object']).columns.tolist()
print(object_cols)

if len(object_cols) > 0:
    print("\n🔧 Sample values in object columns:")
    for col in object_cols[:5]:  # Show first 5 object columns
        print(f"   {col}: {X[col].unique()[:5]}")

# Prepare X for mutual information by handling object columns
X_for_analysis = X.copy()

# Convert object columns to numeric using label encoding for MI analysis
from sklearn.preprocessing import LabelEncoder

if len(object_cols) > 0:
    print(f"\n⚙️ Converting {len(object_cols)} object columns to numeric for analysis...")
    label_encoders = {}
    
    for col in object_cols:
        le = LabelEncoder()
        X_for_analysis[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    print("✅ Object columns converted to numeric")

# Now calculate Mutual Information for feature importance
print("\n🧠 Calculating Mutual Information...")
mi_scores = mutual_info_classif(X_for_analysis, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': X.columns,
    'Mutual_Information': mi_scores
}).sort_values('Mutual_Information', ascending=False)

print("📊 Top 15 Features by Mutual Information:")
print("=" * 50)
for i, (_, row) in enumerate(mi_df.head(15).iterrows(), 1):
    print(f"{i:2d}. {row['Feature']:<30}: {row['Mutual_Information']:.6f}")

# Visualize top features
top_features = mi_df.head(15)
fig_mi = px.bar(
    top_features, 
    x='Mutual_Information', 
    y='Feature', 
    orientation='h',
    title='🧠 Top 15 Features by Mutual Information',
    color='Mutual_Information',
    color_continuous_scale='viridis'
)
fig_mi.update_layout(
    height=600,
    title_x=0.5,
    yaxis={'categoryorder': 'total ascending'}
)
fig_mi.show()

🔍 Checking data types in X:
bool       22
float64    17
object      8
Name: count, dtype: int64

📋 Object columns found:
['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

🔧 Sample values in object columns:
   Gender: ['Male' 'Female']
   family_history_with_overweight: ['yes' 'no']
   FAVC: ['yes' 'no']
   CAEC: ['Sometimes' 'Frequently' 'no' 'Always']
   SMOKE: ['no' 'yes']

⚙️ Converting 8 object columns to numeric for analysis...
✅ Object columns converted to numeric

🧠 Calculating Mutual Information...
📊 Top 15 Features by Mutual Information:
 1. BMI                           : 1.389245
 2. Weight                        : 1.370166
 3. Weight_scaled                 : 1.365949
 4. Age                           : 0.802202
 5. Age_scaled                    : 0.799001
 6. Height                        : 0.782180
 7. Height_scaled                 : 0.777822
 8. CH2O                          : 0.526750
 9. CH2O_scaled                   : 0.524

### 🌲 Random Forest Feature Importance

In [7]:
# Calculate Random Forest feature importance
print("🌲 Calculating Random Forest feature importance...")

# Use the same processed data from MI analysis (X_for_analysis should still be available)
# If not available, recreate it
if 'X_for_analysis' not in locals():
    X_for_analysis = X.copy()
    object_cols = X.select_dtypes(include=['object']).columns.tolist()
    
    if len(object_cols) > 0:
        print(f"   Converting {len(object_cols)} object columns to numeric...")
        for col in object_cols:
            le = LabelEncoder()
            X_for_analysis[col] = le.fit_transform(X[col].astype(str))

# Train Random Forest on processed data
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_for_analysis, y)

rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'RF_Importance': rf.feature_importances_
}).sort_values('RF_Importance', ascending=False)

print("🌲 Top 15 Features by Random Forest Importance:")
print("=" * 55)
for i, (_, row) in enumerate(rf_importance.head(15).iterrows(), 1):
    print(f"{i:2d}. {row['Feature']:<30}: {row['RF_Importance']:.6f}")

# Visualize Random Forest importance
top_rf_features = rf_importance.head(15)
fig_rf = px.bar(
    top_rf_features, 
    x='RF_Importance', 
    y='Feature', 
    orientation='h',
    title='🌲 Top 15 Features by Random Forest Importance',
    color='RF_Importance',
    color_continuous_scale='plasma'
)
fig_rf.update_layout(
    height=600,
    title_x=0.5,
    yaxis={'categoryorder': 'total ascending'}
)
fig_rf.show()

🌲 Calculating Random Forest feature importance...
🌲 Top 15 Features by Random Forest Importance:
 1. BMI                           : 0.194123
 2. Weight                        : 0.147674
 3. Weight_scaled                 : 0.140554
 4. FCVC_scaled                   : 0.043868
 5. Age                           : 0.041752
 6. Age_scaled                    : 0.039538
 7. Height                        : 0.038439
 8. Height_scaled                 : 0.035725
 9. FCVC                          : 0.027481
10. Gender                        : 0.023473
11. Gender_Female                 : 0.022864
12. Gender_Male                   : 0.021949
13. CH2O_scaled                   : 0.019289
14. CH2O                          : 0.018344
15. FAF                           : 0.015622
🌲 Top 15 Features by Random Forest Importance:
 1. BMI                           : 0.194123
 2. Weight                        : 0.147674
 3. Weight_scaled                 : 0.140554
 4. FCVC_scaled                   : 0.043868
 

### 🎯 Combined Feature Importance Analysis

In [8]:
# Combine both importance measures
combined_importance = mi_df.merge(rf_importance, on='Feature')

# Normalize scores to 0-1 range for fair comparison
combined_importance['MI_Normalized'] = (
    combined_importance['Mutual_Information'] / 
    combined_importance['Mutual_Information'].max()
)
combined_importance['RF_Normalized'] = (
    combined_importance['RF_Importance'] / 
    combined_importance['RF_Importance'].max()
)

# Calculate combined score (average of normalized scores)
combined_importance['Combined_Score'] = (
    combined_importance['MI_Normalized'] + 
    combined_importance['RF_Normalized']
) / 2

combined_importance = combined_importance.sort_values('Combined_Score', ascending=False)

print("🎯 Top 20 Features by Combined Importance (MI + RF):")
print("=" * 70)
print(f"{'Rank':<4} {'Feature':<30} {'Combined':<10} {'MI':<10} {'RF':<10}")
print("-" * 70)

for i, (_, row) in enumerate(combined_importance.head(20).iterrows(), 1):
    print(f"{i:<4} {row['Feature']:<30} {row['Combined_Score']:.6f}  "
          f"{row['MI_Normalized']:.6f} {row['RF_Normalized']:.6f}")

# Visualize combined importance
top_combined = combined_importance.head(20)
fig_combined = px.bar(
    top_combined, 
    x='Combined_Score', 
    y='Feature', 
    orientation='h',
    title='🎯 Top 20 Features by Combined Importance (MI + RF)',
    color='Combined_Score',
    color_continuous_scale='turbo'
)
fig_combined.update_layout(
    height=700,
    title_x=0.5,
    yaxis={'categoryorder': 'total ascending'}
)
fig_combined.show()

🎯 Top 20 Features by Combined Importance (MI + RF):
Rank Feature                        Combined   MI         RF        
----------------------------------------------------------------------
1    BMI                            1.000000  1.000000 1.000000
2    Weight                         0.873494  0.986267 0.760722
3    Weight_scaled                  0.853637  0.983232 0.724043
4    Age                            0.396259  0.577438 0.215080
5    Age_scaled                     0.389404  0.575134 0.203675
6    Height                         0.380520  0.563025 0.198015
7    Height_scaled                  0.371961  0.559889 0.184034
8    FCVC_scaled                    0.293801  0.361621 0.225981
9    FCVC                           0.252935  0.364305 0.141566
10   CH2O_scaled                    0.238331  0.377299 0.099362
11   CH2O                           0.236829  0.379163 0.094495
12   TUE                            0.219109  0.359771 0.078448
13   TUE_scaled                     0.21

### 📈 Feature Selection Strategy

In [9]:
# Select top features using combined importance
n_features_to_select = min(25, len(X.columns))  # Select top 25 features or all if fewer

selected_features = combined_importance.head(n_features_to_select)['Feature'].tolist()

print(f"🎯 Selected {len(selected_features)} features for model training:")
print("=" * 60)

for i, feature in enumerate(selected_features, 1):
    importance_score = combined_importance[combined_importance['Feature'] == feature]['Combined_Score'].iloc[0]
    print(f"{i:2d}. {feature:<35} (Score: {importance_score:.6f})")

# Create dataset with selected features
X_selected = X[selected_features].copy()

print(f"\n✅ Feature selection completed:")
print(f"   Original features: {X.shape[1]}")
print(f"   Selected features: {X_selected.shape[1]}")
print(f"   Reduction: {((X.shape[1] - X_selected.shape[1]) / X.shape[1] * 100):.1f}%")

🎯 Selected 25 features for model training:
 1. BMI                                 (Score: 1.000000)
 2. Weight                              (Score: 0.873494)
 3. Weight_scaled                       (Score: 0.853637)
 4. Age                                 (Score: 0.396259)
 5. Age_scaled                          (Score: 0.389404)
 6. Height                              (Score: 0.380520)
 7. Height_scaled                       (Score: 0.371961)
 8. FCVC_scaled                         (Score: 0.293801)
 9. FCVC                                (Score: 0.252935)
10. CH2O_scaled                         (Score: 0.238331)
11. CH2O                                (Score: 0.236829)
12. TUE                                 (Score: 0.219109)
13. TUE_scaled                          (Score: 0.215964)
14. FAF                                 (Score: 0.206389)
15. FAF_scaled                          (Score: 0.199829)
16. Gender                              (Score: 0.157211)
17. Gender_Female            

## ⚖️ Feature Scaling and Normalization

Scale the selected features for optimal model performance.

In [10]:
# Prepare selected features with proper encoding for scaling
print("🔧 Preparing selected features for scaling...")

# Create encoded version of selected features
X_selected_encoded = X_selected.copy()
object_cols_selected = X_selected.select_dtypes(include=['object']).columns.tolist()

print(f"📋 Object columns in selected features: {object_cols_selected}")

# Encode object columns in selected features
if len(object_cols_selected) > 0:
    print(f"⚙️ Encoding {len(object_cols_selected)} object columns in selected features...")
    for col in object_cols_selected:
        le = LabelEncoder()
        X_selected_encoded[col] = le.fit_transform(X_selected[col].astype(str))
    print("✅ Object columns encoded for scaling")

# Prepare for scaling - split data first to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_selected_encoded, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\n📊 Data split for scaling:")
print(f"   Training set: {X_train.shape}")
print(f"   Test set: {X_test.shape}")

# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Feature scaling completed using StandardScaler")
print(f"   Mean of scaled training features: {X_train_scaled.mean():.6f}")
print(f"   Std of scaled training features: {X_train_scaled.std():.6f}")

# Convert back to DataFrame for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_selected.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_selected.columns, index=X_test.index)

print(f"\n📊 Scaling statistics for selected features (training set - first 10):")
scaling_stats = pd.DataFrame({
    'Original_Mean': X_train.mean(),
    'Original_Std': X_train.std(),
    'Scaled_Mean': X_train_scaled_df.mean(),
    'Scaled_Std': X_train_scaled_df.std()
})

print(scaling_stats.head(10))

🔧 Preparing selected features for scaling...
📋 Object columns in selected features: ['Gender', 'family_history_with_overweight', 'CAEC', 'CALC']
⚙️ Encoding 4 object columns in selected features...
✅ Object columns encoded for scaling

📊 Data split for scaling:
   Training set: (16606, 25)
   Test set: (4152, 25)

✅ Feature scaling completed using StandardScaler
   Mean of scaled training features: 0.000000
   Std of scaled training features: 1.000000

📊 Scaling statistics for selected features (training set - first 10):
               Original_Mean  Original_Std   Scaled_Mean  Scaled_Std
BMI                30.236431      8.335998  1.882686e-16     1.00003
Weight             87.863797     26.423177 -3.966477e-16     1.00003
Weight_scaled      -0.000909      1.001682 -3.808160e-17     1.00003
Age                23.839810      5.666853 -2.682827e-16     1.00003
Age_scaled         -0.000351      0.996294  1.369226e-17     1.00003
Height              1.699990      0.087491  7.179879e-16   

## 💾 Save Engineered Features and Artifacts

Save all the processed data and artifacts for the next stages.

In [11]:
# Save the complete engineered dataset
engineered_data_path = Path("../data/engineered_features.csv")
df_engineered.to_csv(engineered_data_path, index=False)

# Save selected features dataset
selected_data_path = Path("../data/selected_features.csv")
X_selected_with_target = X_selected.copy()
X_selected_with_target[target_col] = y
X_selected_with_target.to_csv(selected_data_path, index=False)

# Save the scaler
scaler_path = Path("../models/scaler.pkl")
scaler_path.parent.mkdir(exist_ok=True)
joblib.dump(scaler, scaler_path)

# Save feature names for model deployment
feature_names_path = Path("../models/feature_names.pkl")
joblib.dump(selected_features, feature_names_path)

# Save feature importance analysis
feature_analysis_path = Path("../results/feature_importance.csv")
feature_analysis_path.parent.mkdir(exist_ok=True)
combined_importance.to_csv(feature_analysis_path, index=False)

# Create comprehensive feature engineering summary
feature_engineering_summary = {
    'original_features': X.shape[1],
    'selected_features': len(selected_features),
    'engineered_features_added': len(new_features),
    'feature_selection_method': 'Combined (Mutual Information + Random Forest)',
    'scaling_method': 'StandardScaler',
    'selected_feature_list': selected_features,
    'new_engineered_features': new_features,
    'data_split': {
        'train_size': X_train.shape[0],
        'test_size': X_test.shape[0],
        'test_ratio': 0.2
    },
    'target_info': target_info
}

summary_path = Path("../results/feature_engineering_summary.json")
with open(summary_path, 'w') as f:
    json.dump(feature_engineering_summary, f, indent=2)

print("✅ All artifacts saved successfully:")
print(f"   Engineered features: {engineered_data_path}")
print(f"   Selected features: {selected_data_path}")
print(f"   Scaler: {scaler_path}")
print(f"   Feature names: {feature_names_path}")
print(f"   Feature importance: {feature_analysis_path}")
print(f"   Engineering summary: {summary_path}")

✅ All artifacts saved successfully:
   Engineered features: ..\data\engineered_features.csv
   Selected features: ..\data\selected_features.csv
   Scaler: ..\models\scaler.pkl
   Feature names: ..\models\feature_names.pkl
   Feature importance: ..\results\feature_importance.csv
   Engineering summary: ..\results\feature_engineering_summary.json


## 🎯 Summary and Next Steps

### ✅ What we accomplished:
- **Feature Engineering**: Created meaningful derived features (BMI, Health Risk Score, etc.)
- **Feature Analysis**: Analyzed feature importance using multiple methods
- **Feature Selection**: Selected top features using combined importance scores
- **Data Scaling**: Applied StandardScaler to normalize feature values
- **Data Splitting**: Prepared train/test split for model training
- **Artifact Saving**: Saved all necessary files for next stages

### 🔬 Key Engineering Insights:
- **BMI**: Strong correlation with obesity types (as expected)
- **Health Risk Score**: Composite measure combining multiple lifestyle factors
- **Feature Importance**: Combined MI and RF analysis for robust selection
- **Scaling**: Proper normalization to ensure fair model training

### ➡️ Next Steps:
1. **Model Training** (`04_Model_Training.ipynb`)
   - Load engineered features
   - Apply SMOTE for class balancing
   - Train multiple baseline models
   - Initial performance evaluation

2. **Hyperparameter Tuning** (`05_Hyperparameter_Tuning_Evaluation.ipynb`)
   - Grid search optimization
   - Cross-validation analysis
   - Final model selection
   - Comprehensive evaluation

### 📊 Ready for Training:
- **Selected Features**: 25 most important features
- **Engineered Features**: BMI and composite health indicators
- **Scaled Data**: Properly normalized for model training
- **Data Split**: 80/20 train/test split with stratification

---
**📝 Note**: All feature engineering maintains the original data relationships while adding meaningful derived features that should improve model performance. The scaling and selection processes follow best practices to avoid data leakage.